# Day 3 Session 2: MCMC with Real Data and Forecasting

By the end of this session, you should be able to:

1. Explain why likelihood can be used as a score for model parameters.
2. Run a simple random-walk Metropolis MCMC algorithm.
3. Interpret trace plots, histograms, and acceptance rate.
4. Use sampled parameters in a model to generate a forecast.
5. Compare a model forecast with real data.


> **Participant notebook:** Work through the examples in order. For exercises that ask you to modify an existing example, a copied workspace or starter cell is included so you can change the named values without editing the original demonstration cell.


## 1. Review: likelihood as a model score

For a normal observation model, we can write

$$
y_t \sim N(\text{model prediction at time }t,\ \sigma^2).
$$

This says:

> If the model prediction is close to the observed data, the likelihood is larger.  
> If the model prediction is far from the observed data, the likelihood is smaller.

Today we use the likelihood as a score inside MCMC. We will avoid a full Bayesian statistics lecture and focus on the computation and interpretation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

rng = np.random.default_rng(2026)

## 2. Load the real data

The file `AZ_DATA.csv` contains daily Arizona confirmed influenza hospitalizations from **October 1, 2023** to **April 27, 2024**.

The notebook first looks for `AZ_DATA.csv`. If the file is not available, it creates a small synthetic fallback dataset so the notebook still runs. For the workshop, use the real CSV file.


In [ ]:
github_url = "https://raw.githubusercontent.com/AZDSC-NAU/AZDSC_NAU_SITE/master/project_1/materials/AZ_DATA.csv"
try:
    data = pd.read_csv(github_url)
    print("Loaded CSV from GitHub.")
except Exception as e:
    print(f"Could not load from GitHub: {e}")
    print("Trying local file instead...")
    data = pd.read_csv("AZ_DATA.csv")
    print("Loaded CSV from local file.")

data["date"] = pd.to_datetime(data["date"])
data["hospitalized"] = data["total_patients_hospitalized_confirmed_influenza"].astype(float)

data.head()

## 3. Choose a training period and a forecast period

We will fit the model using the first part of the data and then forecast the remaining days.

- **Training data:** used to estimate parameters.
- **Forecast period:** held out until after the parameters are estimated.

This is a useful classroom habit: do not judge a model only on the data it was fitted to.


In [ ]:
y = data["hospitalized"].to_numpy()
dates = data["date"].to_numpy()
n_days = len(y)

# Use the first 120 days for training and forecast the remaining days.
# Participants can change this number later.
train_end = 120
train_y = y[:train_end]
forecast_y = y[train_end:]

print(f"Number of days: {n_days}")
print(f"Training period: {data['date'].iloc[0].date()} to {data['date'].iloc[train_end-1].date()}")
print(f"Forecast period: {data['date'].iloc[train_end].date()} to {data['date'].iloc[-1].date()}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(data["date"], y, marker="o", markersize=3, linewidth=1, label="Observed hospitalizations")
plt.axvline(data["date"].iloc[train_end], linestyle="--", label="Forecast begins")
plt.title("Arizona Influenza Hospitalizations")
plt.xlabel("Date")
plt.ylabel("Hospitalized patients")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

### Discussion pause

Before modeling, describe the data in words.

1. When does the hospitalization curve increase most quickly?
2. Around when is the peak?
3. Does the curve look perfectly smooth? Why might real data be noisy?


## 4. A simple SIR model for the infection curve

We use the SIR model:

$$
\begin{aligned}
\frac{dS}{dt} &= -\frac{\beta S I}{N}, \\
\frac{dI}{dt} &= \frac{\beta S I}{N} - \gamma I, \\
\frac{dR}{dt} &= \gamma I.
\end{aligned}
$$

The data are hospitalizations, not total infections. To keep the model simple, we assume hospitalizations are proportional to the infected compartment:

$$
H(t) = \rho I(t).
$$

Here:

- $\beta$ controls transmission.
- $\gamma$ controls recovery/removal.
- $\rho$ is a scaling factor from infections to hospitalizations.

To keep the MCMC exercise focused, we will estimate only $\beta$ and $\gamma$. We will fix $\rho$ and the initial infected value $I_0$.


In [ ]:
def sir_predict_hospitalizations(beta, gamma, n_days, rho=0.0014, population=7_600_000, I0=3040.0):
    '''Simulate a simple SIR model and return predicted hospitalizations H(t) = rho * I(t).
    
    This uses a daily Euler step. It is intentionally simple for teaching.
    '''
    S0 = population - I0
    R0 = 0.0
    states = np.zeros((n_days, 3))
    states[0] = np.array([S0, I0, R0])
    
    for day in range(1, n_days):
        S, I, R = states[day - 1]
        N = S + I + R
        
        dS = -beta * S * I / N
        dI = beta * S * I / N - gamma * I
        dR = gamma * I
        
        next_state = states[day - 1] + np.array([dS, dI, dR])
        states[day] = np.maximum(next_state, 0)
    
    predicted_hospitalizations = rho * states[:, 1]
    return predicted_hospitalizations

In [ ]:
# Try one parameter pair before fitting.
example_beta = 0.22
example_gamma = 0.16
example_prediction = sir_predict_hospitalizations(example_beta, example_gamma, n_days)

plt.figure(figsize=(10, 4))
plt.plot(data["date"][:train_end], y[:train_end], marker="o", markersize=3, linewidth=1, label="Observed data")
plt.plot(data["date"][train_end:], y[train_end:], marker="o", markersize=3, linewidth=1, label="Observed data (forecast period)")
plt.plot(data["date"], example_prediction, linewidth=2, label=fr"SIR prediction, $\beta$={example_beta}, $\gamma$={example_gamma}")
plt.axvline(data["date"].iloc[train_end], linestyle="--", label="Forecast begins")
plt.title("Trying One Parameter Pair")
plt.xlabel("Date")
plt.ylabel("Hospitalized patients")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Normal likelihood for the real data

We assume the observed hospitalization count is close to the model prediction, with some noise:

$$
y_t \sim N(H(t;\beta,\gamma), \sigma^2).
$$

Equivalently, parameter values are better when the model curve is closer to the observed training data.

We use the log-likelihood because it is easier and safer to compute than multiplying many small likelihood values.


In [ ]:
# Observation noise used in the normal likelihood.
# This is a teaching choice. You may need to change it for different state or different flu season.
# The larger the value, the more uncertainty is assumed in the data.
# Ideally, you would estimate this by MCMC, but we will keep it fixed for simplicity.
sigma_obs = 75.0


def log_likelihood_sir(theta, y_train, sigma=sigma_obs):
    '''Log-likelihood for beta and gamma using the training data.'''
    beta, gamma = theta
    
    # Keep the search in a biologically reasonable and numerically stable range.
    if beta <= 0 or gamma <= 0 or beta > 0.5 or gamma > 0.5:
        return -np.inf
    
    pred = sir_predict_hospitalizations(beta, gamma, len(y_train))
    residual = y_train - pred
    log_density = -np.log(sigma * np.sqrt(2 * np.pi)) - 0.5 * (residual / sigma) ** 2
    return np.sum(log_density)


def sse_sir(theta, y_train):
    '''Sum of squared errors. Used only to create a good starting point.'''
    beta, gamma = theta
    pred = sir_predict_hospitalizations(beta, gamma, len(y_train))
    return np.sum((y_train - pred) ** 2)

print("Example log-likelihood:", log_likelihood_sir((0.22, 0.16), train_y))

## 6. Starting point for MCMC

MCMC needs a starting value. Since Day 2 already covered grid search, we can use a short grid search to get a reasonable starting point.

This is not the main idea of Session 2; it is just a practical way to help the MCMC chain start in a useful region.


In [ ]:
beta_grid = np.arange(0.06, 0.36, 0.02)
gamma_grid = np.arange(0.04, 0.30, 0.02)

best_theta = None
best_sse = np.inf

for beta in beta_grid:
    for gamma in gamma_grid:
        current_sse = sse_sir((beta, gamma), train_y)
        if current_sse < best_sse:
            best_sse = current_sse
            best_theta = np.array([beta, gamma])

print(f"Grid-search starting point: beta={best_theta[0]:.3f}, gamma={best_theta[1]:.3f}")
print(f"Training SSE at starting point: {best_sse:,.1f}")

## 7. Random-walk Metropolis MCMC

At each step:

1. Start with the current values of $\beta$ and $\gamma$.
2. Propose a nearby pair:

$$
\begin{aligned}
\beta_{proposal} &= \beta_{current} + \text{small random noise},\\
\gamma_{proposal} &= \gamma_{current} + \text{small random noise}.
\end{aligned}
$$

3. Compute the likelihood score for the current and proposed values.
4. Accept the proposal if it improves the likelihood.
5. Sometimes accept worse proposals, so the chain can explore instead of only climbing uphill.


In [ ]:
def random_walk_metropolis_sir(
    y_train,
    initial_theta,
    proposal_sd=(0.002, 0.002), #
    n_iter=3000,
    sigma=sigma_obs,
    seed=2026,
):
    '''Random-walk Metropolis sampler for beta and gamma.'''
    rng = np.random.default_rng(seed)
    chain = np.zeros((n_iter, 2))
    log_scores = np.zeros(n_iter)
    accepted = np.zeros(n_iter, dtype=bool)
    
    current_theta = np.array(initial_theta, dtype=float)
    current_log_score = log_likelihood_sir(current_theta, y_train, sigma=sigma)
    
    chain[0] = current_theta
    log_scores[0] = current_log_score
    
    for i in range(1, n_iter):
        proposal = current_theta + rng.normal(loc=0.0, scale=proposal_sd, size=2)
        proposal_log_score = log_likelihood_sir(proposal, y_train, sigma=sigma)
        
        log_acceptance_ratio = proposal_log_score - current_log_score
        
        if np.log(rng.uniform()) < log_acceptance_ratio:
            current_theta = proposal
            current_log_score = proposal_log_score
            accepted[i] = True
        
        chain[i] = current_theta
        log_scores[i] = current_log_score
    
    acceptance_rate = accepted[1:].mean()
    return chain, log_scores, acceptance_rate

## 8. Run MCMC

The proposal standard deviation controls how large the random-walk steps are.

- Too small: the chain moves slowly.
- Too large: many proposals get rejected.
- A middle value usually works better.


In [ ]:
n_iter = 3000
proposal_sd = (0.002, 0.002)

chain, log_scores, acceptance_rate = random_walk_metropolis_sir(
    y_train=train_y,
    initial_theta=best_theta,
    proposal_sd=proposal_sd,
    n_iter=n_iter,
    sigma=sigma_obs,
    seed=2026,
)

burn_in = 500
post_chain = chain[burn_in:]
post_log_scores = log_scores[burn_in:]

beta_mean, gamma_mean = post_chain.mean(axis=0)
beta_median, gamma_median = np.median(post_chain, axis=0)
beta_interval = np.percentile(post_chain[:, 0], [5, 95])
gamma_interval = np.percentile(post_chain[:, 1], [5, 95])

best_sample_index = np.argmax(post_log_scores)
beta_best, gamma_best = post_chain[best_sample_index]

print(f"Acceptance rate: {acceptance_rate:.3f}")
print(f"Posterior sample mean: beta={beta_mean:.4f}, gamma={gamma_mean:.4f}")
print(f"Posterior sample median: beta={beta_median:.4f}, gamma={gamma_median:.4f}")
print(f"Best sampled value: beta={beta_best:.4f}, gamma={gamma_best:.4f}")
print(f"90% sample interval for beta:  [{beta_interval[0]:.4f}, {beta_interval[1]:.4f}]")
print(f"90% sample interval for gamma: [{gamma_interval[0]:.4f}, {gamma_interval[1]:.4f}]")

## 9. Check the MCMC chain

The trace plot shows where the chain moved over time. A useful chain should move around a stable region instead of drifting forever.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(chain[:, 0], label=r"$\beta$")
plt.plot(chain[:, 1], label=r"$\gamma$")
plt.axvline(burn_in, linestyle="--", label="Burn-in cutoff")
plt.title("Trace Plot for MCMC Samples")
plt.xlabel("Iteration")
plt.ylabel("Parameter value")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(post_chain[:, 0], bins=35, alpha=0.7, label=r"$\beta$")
plt.hist(post_chain[:, 1], bins=35, alpha=0.7, label=r"$\gamma$")
plt.title("Histogram of MCMC Samples After Burn-in")
plt.xlabel("Parameter value")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(post_chain[:, 0], post_chain[:, 1], s=8, alpha=0.35)
plt.scatter(beta_best, gamma_best, s=80, marker="*", label="Best sampled value")
plt.xlabel(r"$\beta$")
plt.ylabel(r"$\gamma$")
plt.title("MCMC Samples for Beta and Gamma")
plt.legend()
plt.tight_layout()
plt.show()

### Interpretation pause

1. What does the acceptance rate tell us?
2. Do the sampled values stay in a reasonable region?
3. Are $\beta$ and $\gamma$ independent, or do they seem related in the scatter plot?


## 10. Substitute the sampled parameters back into the model

Now we use the sampled parameters to generate model curves.

We will compare:

1. The **posterior mean parameters**.
2. The **best sampled parameters**.
3. A small collection of sampled parameter curves to show uncertainty.

This is the key step connecting MCMC back to forecasting.


In [ ]:
mean_prediction = sir_predict_hospitalizations(beta_mean, gamma_mean, n_days)
best_prediction = sir_predict_hospitalizations(beta_best, gamma_best, n_days)

train_rmse = np.sqrt(np.mean((train_y - mean_prediction[:train_end]) ** 2))
forecast_rmse = np.sqrt(np.mean((forecast_y - mean_prediction[train_end:]) ** 2))

print(f"Training RMSE using posterior mean parameters: {train_rmse:.2f}")
print(f"Forecast RMSE using posterior mean parameters: {forecast_rmse:.2f}")

In [ ]:
# Generate several forecast curves from MCMC samples.
# This gives a simple visual sense of parameter uncertainty.
num_curves = 60
sample_indices = rng.choice(len(post_chain), size=num_curves, replace=False)
sample_predictions = np.zeros((num_curves, n_days))

for k, idx in enumerate(sample_indices):
    beta_k, gamma_k = post_chain[idx]
    sample_predictions[k] = sir_predict_hospitalizations(beta_k, gamma_k, n_days)

lower_curve = np.percentile(sample_predictions, 10, axis=0)
upper_curve = np.percentile(sample_predictions, 90, axis=0)

In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(data["date"][:train_end], y[:train_end], marker="o", markersize=3, linewidth=1, label="Observed data")
plt.plot(data["date"][train_end:], y[train_end:], marker="o", markersize=3, linewidth=1, label="Observed data (forecast period)")
plt.plot(data["date"], mean_prediction, linewidth=2.5, label="Forecast using posterior mean parameters")
plt.plot(data["date"], best_prediction, linewidth=2, linestyle="--", label="Forecast using best sampled parameters")
plt.fill_between(data["date"], lower_curve, upper_curve, alpha=0.2, label="Middle 80% of sampled model curves")
plt.axvline(data["date"].iloc[train_end], linestyle="--", label="Forecast begins")

plt.title("MCMC-Fitted SIR Model Forecast vs Real Data")
plt.xlabel("Date")
plt.ylabel("Hospitalized patients")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

### Forecast discussion

1. Does the model capture the timing of the peak?
2. Does it over-predict or under-predict the forecast period?
3. Does the uncertainty band cover the real data well?
4. What important real-world features are missing from this simple SIR model?


## 11. Exercise 1: Modify the MCMC proposal size

Change `proposal_sd` below and rerun the cell.

Try:

- `(0.0005, 0.0005)`
- `(0.002, 0.002)`
- `(0.01, 0.01)`
- `(0.03, 0.03)`

Discuss:

1. What happens to the acceptance rate?
2. What happens to the trace plot?
3. Which proposal size gives a useful balance between moving and accepting?


In [ ]:
# Exercise 1 workspace
proposal_sd_exercise = (0.002, 0.002)   # Change this
n_iter_exercise = 1500                  # You can also change this

exercise_chain, exercise_log_scores, exercise_acceptance_rate = random_walk_metropolis_sir(
    y_train=train_y,
    initial_theta=best_theta,
    proposal_sd=proposal_sd_exercise,
    n_iter=n_iter_exercise,
    sigma=sigma_obs,
    seed=123,
)

print(f"Acceptance rate: {exercise_acceptance_rate:.3f}")
print(f"Mean beta after burn-in:  {exercise_chain[300:, 0].mean():.4f}")
print(f"Mean gamma after burn-in: {exercise_chain[300:, 1].mean():.4f}")

plt.figure(figsize=(10, 4))
plt.plot(exercise_chain[:, 0], label=r"$\beta$")
plt.plot(exercise_chain[:, 1], label=r"$\gamma$")
plt.axvline(300, linestyle="--", label="Burn-in cutoff")
plt.title("Exercise Trace Plot")
plt.xlabel("Iteration")
plt.ylabel("Parameter value")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Exercise 2: Change the training period

Change `train_end_exercise` and rerun the cell.

Try:

- `train_end_exercise = 80`
- `train_end_exercise = 120`
- `train_end_exercise = 150`

Discuss:

1. What happens if we forecast before seeing the peak?
2. What happens if we train on more of the season?
3. Is it fair to compare models only on the training period?


In [ ]:
# Exercise 2 workspace
train_end_exercise = 120  # Change this to 80, 120, or 150
train_y_exercise = y[:train_end_exercise]
forecast_y_exercise = y[train_end_exercise:]

# Use the same starting point to keep the exercise short.
exercise_chain2, exercise_log_scores2, exercise_acceptance_rate2 = random_walk_metropolis_sir(
    y_train=train_y_exercise,
    initial_theta=best_theta,
    proposal_sd=(0.002, 0.002),
    n_iter=1500,
    sigma=sigma_obs,
    seed=456,
)

exercise_post2 = exercise_chain2[300:]
beta_ex2, gamma_ex2 = exercise_post2.mean(axis=0)
exercise_prediction2 = sir_predict_hospitalizations(beta_ex2, gamma_ex2, n_days)

train_rmse2 = np.sqrt(np.mean((train_y_exercise - exercise_prediction2[:train_end_exercise]) ** 2))
forecast_rmse2 = np.sqrt(np.mean((forecast_y_exercise - exercise_prediction2[train_end_exercise:]) ** 2))

print(f"Acceptance rate: {exercise_acceptance_rate2:.3f}")
print(f"Mean beta: {beta_ex2:.4f}, mean gamma: {gamma_ex2:.4f}")
print(f"Training RMSE: {train_rmse2:.2f}")
print(f"Forecast RMSE: {forecast_rmse2:.2f}")

plt.figure(figsize=(11, 5))
plt.plot(data["date"], y, marker="o", markersize=3, linewidth=1, label="Observed data")
plt.plot(data["date"], exercise_prediction2, linewidth=2.5, label="MCMC forecast")
plt.axvline(data["date"].iloc[train_end_exercise], linestyle="--", label="Forecast begins")
plt.title("Changing the Training Period")
plt.xlabel("Date")
plt.ylabel("Hospitalized patients")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 13. Exercise 3: Change the observation noise

The value `sigma_obs` controls how strongly the likelihood penalizes model-data mismatch.

Try:

- `sigma_exercise = 40`
- `sigma_exercise = 75`
- `sigma_exercise = 150`

Discuss:

1. Does smaller observation noise make the likelihood stricter or more forgiving?
2. Does larger observation noise make the MCMC samples more spread out?
3. How could we estimate observation noise in a more advanced model?


In [ ]:
# Exercise 3 workspace
sigma_exercise = 75.0  # Change this

exercise_chain3, exercise_log_scores3, exercise_acceptance_rate3 = random_walk_metropolis_sir(
    y_train=train_y,
    initial_theta=best_theta,
    proposal_sd=(0.002, 0.002),
    n_iter=1500,
    sigma=sigma_exercise,
    seed=789,
)

exercise_post3 = exercise_chain3[300:]
print(f"Acceptance rate: {exercise_acceptance_rate3:.3f}")
print(f"SD of beta samples:  {exercise_post3[:, 0].std():.5f}")
print(f"SD of gamma samples: {exercise_post3[:, 1].std():.5f}")

plt.figure(figsize=(8, 4))
plt.hist(exercise_post3[:, 0], bins=30, alpha=0.7, label=r"$\beta$")
plt.hist(exercise_post3[:, 1], bins=30, alpha=0.7, label=r"$\gamma$")
plt.title("Effect of Observation Noise on MCMC Samples")
plt.xlabel("Parameter value")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## Session 2 wrap-up

Today we completed the full loop:

$$
\text{real data} \rightarrow \text{likelihood} \rightarrow \text{MCMC samples} \rightarrow \text{model forecast} \rightarrow \text{comparison with data}.
$$

Main takeaways:

1. The likelihood gives a way to score parameter values.
2. MCMC uses random proposals and accept/reject decisions to explore good parameter regions.
3. MCMC samples can be substituted back into the model to generate forecasts.
4. Real-data forecasting is harder than fitting a smooth classroom curve, so model checking is essential.
